# DPO: Direct Preference Optimization - 실습 코드 2: DPO Loss 직접 구현 (PyTorch)

- Tutorial ID: `expand-dpo`
- Tutorial: DPO: Direct Preference Optimization
- Section ID: `expand-dpo-code-2`
- Section: 실습 코드 2: DPO Loss 직접 구현 (PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: DPO Loss 직접 구현 (PyTorch)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DPOLoss(nn.Module):
    """DPO (Direct Preference Optimization) Loss 직접 구현
    
    논문: Rafailov et al., "Direct Preference Optimization", NeurIPS 2023
    
    핵심 수식:
    L_DPO = -E[log σ(β · log(π_θ(y_w|x)/π_ref(y_w|x)) 
                          - β · log(π_θ(y_l|x)/π_ref(y_l|x)))]
    """
        def __init__(self, beta=0.1, loss_type="sigmoid"):
        super().__init__()
        self.beta = beta
        self.loss_type = loss_type
    
        def forward(self, policy_chosen_logps, policy_rejected_logps,
                reference_chosen_logps, reference_rejected_logps):
        """
        policy_chosen_logps:    log π_θ(y_w|x) — 정책의 chosen 로그 확률
        policy_rejected_logps:  log π_θ(y_l|x) — 정책의 rejected 로그 확률
        reference_chosen_logps: log π_ref(y_w|x) — 참조 모델의 chosen 로그 확률
        reference_rejected_logps: log π_ref(y_l|x) — 참조 모델의 rejected 로그 확률
        """
        # 로그 비율 계산: log(π_θ / π_ref)
        chosen_logratios = policy_chosen_logps - reference_chosen_logps
        rejected_logratios = policy_rejected_logps - reference_rejected_logps
        
        # DPO Loss
                logits = self.beta * (chosen_logratios - rejected_logratios)
        
        if self.loss_type == "sigmoid":
                        loss = -F.logsigmoid(logits).mean()
        elif self.loss_type == "hinge":
                        loss = F.relu(1 - logits).mean()
        
        # 보조 지표 계산
        with torch.no_grad():
            # 선택 정확도 (chosen이 rejected보다 높은 확률)
            accuracy = (logits > 0).float().mean()
            # 보상 마진
            reward_margin = (chosen_logratios - rejected_logratios).mean()
        
        return loss, accuracy, reward_margin

# ── DPO 학습 루프 ──
def train_dpo(policy_model, ref_model, tokenizer, 
              preference_data, epochs=3, beta=0.1, lr=5e-7):
    """DPO 학습 전체 루프"""
    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=lr)
    dpo_loss_fn = DPOLoss(beta=beta)
    
    # 참조 모델은 동결
    ref_model.eval()
        for param in ref_model.parameters():
        param.requires_grad = False
    
    policy_model.train()
    
        for epoch in range(epochs):
                total_loss = 0
        total_acc = 0
        total_margin = 0
        
                for prompt, chosen_text, rejected_text in preference_data:
            # 토큰화
            chosen_ids = tokenizer(
                f"User: {prompt}\nAssistant: {chosen_text}",
                return_tensors="pt"
            )["input_ids"]
            rejected_ids = tokenizer(
                f"User: {prompt}\nAssistant: {rejected_text}",
                return_tensors="pt"
            )["input_ids"]
            
            # 정책 모델 로그 확률
            chosen_logps = -policy_model(chosen_ids, labels=chosen_ids).loss
            rejected_logps = -policy_model(rejected_ids, labels=rejected_ids).loss
            
            # 참조 모델 로그 확률 (그래디언트 불필요)
            with torch.no_grad():
                ref_chosen_logps = -ref_model(chosen_ids, labels=chosen_ids).loss
                ref_rejected_logps = -ref_model(rejected_ids, labels=rejected_ids).loss
            
            # DPO Loss 계산
            loss, acc, margin = dpo_loss_fn(
                chosen_logps, rejected_logps,
                ref_chosen_logps, ref_rejected_logps
            )
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            total_acc += acc.item()
            total_margin += margin.item()
        
        n = len(preference_data)
                print(f"Epoch {epoch+1}: loss={total_loss/n:.4f}, "
              f"acc={total_acc/n*100:.1f}%, margin={total_margin/n:.4f}")
    
    return policy_model

# ── DPO vs RLHF 비교 ──
print("=== DPO vs RLHF 비교 ===\n")
print("RLHF: SFT → Reward Model → PPO (3단계, 4개 모델)")
print("DPO:  SFT → DPO 직접 최적화 (2단계, 2개 모델)\n")

print("DPO 수학적 핵심:")
print("  보상 모델의 폐쇄형 해를 이용:")
print("  r(x,y) = β·log(π*/π_ref) + β·log Z(x)")
print("  → 보상 모델 없이 정책을 직접 최적화!\n")

print("장점:")
print("  ✅ 보상 모델 불필요 (메모리 절약)")
print("  ✅ PPO 불안정성 없음 (분류 손실)")
print("  ✅ 구현이 단순하고 재현성 높음")
print("  ✅ 보상 해킹(Reward Hacking) 불가")